# Réplicas pareadas pos_weight=2 para F2 — LSTM32 Colab CPU

Se comparan dos entrenamientos desde cero de LSTM32 con dropout 0,2. Solo cambia la pérdida: BCE estándar frente a BCE con pos_weight=2 para la clase positiva. Ambos seleccionan checkpoint y umbral por F2; datos, semilla, Adam, tasa de aprendizaje y runtime CPU se mantienen fijos. Esta ejecución replica la comparación favorable de semilla 42 con semillas 43 y 44. Ambas usan el mismo runtime CPU y conservarán sus artefactos separados dentro del ZIP final. El test 2025 permanece reservado.


## 1. Subir y verificar el paquete CPU

El paquete ya está listo como `runs/lstm_colab_pos_weight2_cpu_replicates_43_44_v1.zip`. Para regenerarlo desde la raíz del repositorio, ejecuta `python3.13 scripts/empaquetar_pos_weight_replicates.py --output runs/lstm_colab_pos_weight2_cpu_replicates_43_44_v1.zip`. Sube el ZIP al panel **Archivos** de Colab cuando la primera celda lo solicite. El SHA-256 esperado está fijado en el notebook; no lo reemplaces con otro paquete.


In [ ]:
from pathlib import Path
import hashlib, json, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = Path('/content/f1-lstm-pos-weight2-replicates-cpu-v1')
PACKAGE = Path('/content/lstm_colab_pos_weight2_cpu_replicates_43_44_v1.zip')
EXPECTED_PACKAGE_SHA256 = '63acd8654aa452f75280dff055b8fa8fe02013b9049ffda845337ff414ca7659'
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-lstm-pos-weight2-replicates-cpu-v1 ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP CPU del par LSTM BCE normal/ponderada.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != 'lstm_colab_pos_weight2_cpu_replicates_43_44_v1.zip': raise ValueError('Se requiere lstm_colab_pos_weight2_cpu_replicates_43_44_v1.zip; no uses el ZIP GPU ni otro archivo.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP CPU corregido.')
with zipfile.ZipFile(PACKAGE) as z:
    members=z.infolist(); names=set()
    for m in members:
        p=Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest=json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_pos_weight_replicates_cpu_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    expected=set(manifest['files_sha256']) | {'colab_package_manifest.json'}
    if names != expected: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT/name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')


## 2. Instalar dependencias y confirmar runtime CPU

Esta variante fija `device=cpu` en ambas configuraciones. No necesita GPU/CUDA; aunque Colab muestre una GPU disponible, el entrenamiento seguirá ejecutándose en CPU. La corrida puede tardar más que la pareja GPU.


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-pos-weight2-replicates-cpu-v1')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete CPU extraído en /content/f1-lstm-pos-weight2-replicates-cpu-v1. Ejecuta primero la sección 1 desde una sesión Colab activa.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab-experiments.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available(), '| dispositivo efectivo de ambas corridas: CPU')


## 3. Ejecutar las dos réplicas pareadas

Para cada semilla (43 y 44), el runner entrena desde cero una referencia BCE normal y una candidata con `pos_weight=2`. Ambas seleccionan checkpoint y umbral por F2; arquitectura, datos y CPU permanecen fijos. El gate comprueba que solo cambie el peso de clase dentro de cada pareja. Al final, se reúnen las dos corridas en un ZIP. La validación 2024 es de desarrollo; 2025 sigue reservada.


In [ ]:
import json, subprocess, sys, zipfile
from pathlib import Path
from google.colab import files
ROOT=Path('/content/f1-lstm-pos-weight2-replicates-cpu-v1')
if not (ROOT/'scripts/entrenar_comparacion_pos_weight.py').is_file(): raise RuntimeError('Extrae primero el paquete de réplicas.')
SEEDS=[43,44]
RUNS={}
for seed in SEEDS:
    RUN=ROOT/f'runs/lstm32_pos_weight2_f2_cpu_v1_seed{seed}'
    if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError(f'La salida seed {seed} ya existe; usa una sesión limpia.')
    reference=f'configs/experimento_lstm32_f2_ref_pos_weight_cpu_seed{seed}_v1.json'
    candidate=f'configs/experimento_lstm32_f2_pos_weight2_cpu_seed{seed}_v1.json'
    result=subprocess.run([sys.executable,'scripts/entrenar_comparacion_pos_weight.py','--reference-config',reference,'--candidate-config',candidate,'--output',str(RUN.relative_to(ROOT))],cwd=ROOT,check=False)
    if result.returncode:
        status=RUN/'status.json'
        if status.is_file(): print(json.dumps(json.loads(status.read_text()),indent=2))
        raise RuntimeError(f'Falló seed {seed}; conserva los artefactos antes de reintentar.')
    summary=json.loads((RUN/'pair_summary.json').read_text())
    RUNS[seed]=RUN
    print(f"seed {seed}: F2 BCE={summary['reference']['validation']['selected_threshold']['f2']:.4f}; F2 pos_weight={summary['candidate']['validation']['selected_threshold']['f2']:.4f}; ΔF2={summary['delta_candidate_minus_reference']['validation_selected_threshold']['f2']:+.4f}")
BUNDLE=ROOT/'runs/lstm32_pos_weight2_f2_cpu_replicates_43_44.zip'
if BUNDLE.exists(): raise FileExistsError('ZIP conjunto ya existe; usa una sesión limpia.')
import hashlib
bundle_hashes={}
with zipfile.ZipFile(BUNDLE,'x',zipfile.ZIP_DEFLATED) as archive:
    for seed,run_dir in RUNS.items():
        for path in sorted(run_dir.rglob('*')):
            if path.is_file() and path.suffix.lower()!='.zip':
                name=path.relative_to(ROOT).as_posix(); archive.write(path,name)
                h=hashlib.sha256(path.read_bytes()).hexdigest(); bundle_hashes[name]=h
    manifest={'artifact_version':'lstm_pos_weight_replicates_v1','seeds':[43,44],'test_used':False,'files_sha256':bundle_hashes}
    archive.writestr('replicate_manifest.json',json.dumps(manifest,indent=2)+'\n')
print('ZIP de réplicas listo:',BUNDLE,'| archivos:',len(bundle_hashes))
files.download(str(BUNDLE))


## 4. Devolver resultados al proyecto

Descarga `lstm32_pos_weight2_f2_cpu_replicates_43_44.zip` y guárdalo intacto en `runs/`. Incluye las parejas seed43 y seed44, cada una con manifiesto e hashes propios. 2025 permanece reservada.
